# Avance 2: Preprocesamiento de Variables, Partición y Entrenamiento de Modelos ML (Unidad 3)
**Proyecto Integrador:** Autolavado Expréss La Estancia  
**Ponderación:** 15 puntos de los 40 del Proyecto Integrador  

### 📌 Objetivos y Entregables del Avance 2:
1. **Preprocesamiento e Ingeniería de Características:** Imputación de nulos (`SimpleImputer`), estandarización / escalamiento de numéricas (`StandardScaler`) y codificación One-Hot de categóricas (`OneHotEncoder`).
2. **Partición de Datos:** División formal Train / Test (80/20) y Validación Cruzada K-Fold ($K=5$).
3. **Entrenamiento de Mínimo 3 Modelos de la Unidad 3:**
   - **Modelos de Regresión / Series de Tiempo:** Regresión Lineal, Árbol de Decisión (CART), Random Forest, Gradient Boosting y KNN Regresor.
   - **Modelos de Clasificación:** Naive Bayes (GaussianNB), KNN (k-Nearest Neighbors), Árbol de Decisión y Random Forest Classifier.
4. **Tabla Comparativa de Rendimiento Preliminar:**
   - Regresión: MAE, RMSE, $R^2$ Score y K-Fold CV Score.
   - Clasificación: Accuracy, Precision, Recall y F1-Score.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

# 1. Conexión a SQLite y lectura de los 103,309 registros históricos
conn = sqlite3.connect('../services/tablero/autolavado.db')
df_raw = pd.read_sql_query('SELECT id_ticket, fecha, dia_semana, hora, tipo_servicio, precio FROM ventas;', conn)
conn.close()

print('Total registros cargados:', len(df_raw))
df_raw.head()

## Parte 1: Modelos de Regresión y Series de Tiempo (Demanda Diaria e Ingresos)
Generamos variables de estacionalidad, calendario y medias móviles para modelar la demanda de vehículos.

In [ ]:
df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])

# Agrupación diaria
daily = df_raw.groupby('fecha').agg(
    vehiculos=('precio', 'count'),
    ingresos=('precio', 'sum')
).reset_index().sort_values('fecha').reset_index(drop=True)

# Rango continuo de fechas
idx_full = pd.date_range(daily['fecha'].min(), daily['fecha'].max(), freq='D')
daily = daily.set_index('fecha').reindex(idx_full, fill_value=0).reset_index().rename(columns={'index': 'fecha'})

# Feature Engineering
daily['dia_nombre'] = daily['fecha'].dt.day_name()
daily['dia_mes'] = daily['fecha'].dt.day
daily['mes'] = daily['fecha'].dt.month
daily['es_fin_semana'] = daily['fecha'].dt.dayofweek.isin([4, 5, 6]).astype(int)
daily['es_quincena'] = daily['dia_mes'].isin([14, 15, 16, 29, 30, 31, 1]).astype(int)
daily['media_movil_7d'] = daily['vehiculos'].shift(1).rolling(7, min_periods=1).mean()
daily['media_movil_14d'] = daily['vehiculos'].shift(1).rolling(14, min_periods=1).mean()

daily_clean = daily.dropna().reset_index(drop=True)
print('Días procesados:', len(daily_clean))
daily_clean.head()

### Preprocesamiento y Partición de Datos (80% Train, 20% Test)
Implementación de `ColumnTransformer` con `SimpleImputer`, `StandardScaler` y `OneHotEncoder`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, cross_val_score

num_cols = ['dia_mes', 'mes', 'es_fin_semana', 'es_quincena', 'media_movil_7d', 'media_movil_14d']
cat_cols = ['dia_nombre']

X = daily_clean[num_cols + cat_cols]
y = daily_clean['vehiculos']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, shuffle=False)
print(f'Train shape: {X_train.shape} | Test shape: {X_test.shape}')

### Entrenamiento y Tabla Comparativa de 5 Modelos de Regresión (Unidad 3)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

models_reg = {
    'Regresión Lineal': LinearRegression(),
    'Árbol de Decisión (CART)': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    'KNN Regresor': KNeighborsRegressor(n_neighbors=7)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
res_reg = []

for name, m in models_reg.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', m)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    cv_score = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2').mean()
    res_reg.append({
        'Algoritmo': name,
        'MAE (Autos)': round(mae, 2),
        'RMSE (Autos)': round(rmse, 2),
        'R² Score (Test)': round(r2, 4),
        'K-Fold CV R² (Train)': round(cv_score, 4)
    })

df_res_reg = pd.DataFrame(res_reg).sort_values(by='R² Score (Test)', ascending=False).reset_index(drop=True)
print('=== TABLA COMPARATIVA DE RENDIMIENTO PRELIMINAR (REGRESIÓN) ===')
display(df_res_reg)

## Parte 2: Modelos de Clasificación de la Unidad 3
Modelos entrenados: **Naive Bayes, KNN, Árboles de Decisión y Random Forest**.
Métricas: **Accuracy, Precision, Recall y F1-Score**.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Muestra estratificada
sample_df = df_raw.sample(n=25000, random_state=42).copy()
sample_df['hora_num'] = sample_df['hora'].apply(lambda x: int(str(x).split(':')[0]))
sample_df['es_fin_semana'] = sample_df['dia_semana'].isin(['Viernes', 'Sábado', 'Domingo']).astype(int)
sample_df['target_detallado'] = sample_df['tipo_servicio'].str.contains('Detallado|Cuatrimoto|Can-Am').astype(int)

X_clf = sample_df[['hora_num', 'precio', 'es_fin_semana', 'dia_semana']]
y_clf = sample_df['target_detallado']

clf_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), ['hora_num', 'precio', 'es_fin_semana']),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(drop='first', sparse_output=False))]), ['dia_semana'])
])

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf)

clf_models = {
    'Naive Bayes (GaussianNB)': GaussianNB(),
    'KNN (k-Nearest Neighbors)': KNeighborsClassifier(n_neighbors=7),
    'Árbol de Decisión (CART)': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42)
}

res_clf = []
for name, m in clf_models.items():
    pipe_clf = Pipeline([('prep', clf_preprocessor), ('clf', m)])
    pipe_clf.fit(X_tr_c, y_tr_c)
    preds = pipe_clf.predict(X_te_c)
    res_clf.append({
        'Algoritmo': name,
        'Accuracy': round(accuracy_score(y_te_c, preds), 4),
        'Precision': round(precision_score(y_te_c, preds, zero_division=0), 4),
        'Recall': round(recall_score(y_te_c, preds, zero_division=0), 4),
        'F1-Score': round(f1_score(y_te_c, preds, zero_division=0), 4)
    })

df_res_clf = pd.DataFrame(res_clf).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print('=== TABLA COMPARATIVA DE RENDIMIENTO PRELIMINAR (CLASIFICACIÓN) ===')
display(df_res_clf)

## Visualización Comparativa de Modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Regresión
sns.barplot(data=df_res_reg, x='Algoritmo', y='R² Score (Test)', palette='mako', ax=axes[0])
axes[0].set_title('Comparativa R² (Regresión - Demanda Diaria)', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=25, ha='right')
axes[0].set_ylim(0, 0.7)

# Clasificación
df_clf_melt = pd.melt(df_res_clf, id_vars=['Algoritmo'], value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'], var_name='Métrica', value_name='Valor')
sns.barplot(data=df_clf_melt, x='Algoritmo', y='Valor', hue='Métrica', palette='viridis', ax=axes[1])
axes[1].set_title('Comparativa Multimétrica (Clasificación Unidad 3)', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')
axes[1].set_ylim(0, 1.15)

plt.tight_layout()
plt.show()